# Polizia di Stato preprocessing

This notebook loads the parsed `polizia_di_stato` MCQ JSONL file, keeps only `CULTURA GENERALE` and null-section examples, removes English and mathematics sections, cleans malformed PDF artifacts, deduplicates examples, inspects the result, and saves an SFT-compatible JSONL file under:

`multilingual/data/processed/polizia_di_stato_clean/polizia_di_stato_clean.jsonl`

In [1]:
from pathlib import Path
import json
import re
from collections import Counter, defaultdict

import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

REPO_ROOT = Path.cwd()
while REPO_ROOT.name != "standard-project-m2-mystery-machine" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

RAW_PATH = REPO_ROOT / "multilingual/data/raw/polizia_di_stato/polizia_di_stato_parsed.jsonl"

OUTPUT_DIR = REPO_ROOT / "multilingual/data/processed/polizia_di_stato_clean"
OUTPUT_PATH = OUTPUT_DIR / "polizia_di_stato_clean.jsonl"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET = "polizia_di_stato_clean"
LANGUAGE = "italian"
SPLIT = "all"
VALID_ANSWERS = set("ABCDE")

KEEP_SECTIONS = {
    None,
    "",
    "CULTURA GENERALE",
}

DROP_SECTIONS = {
    "INGLESE",
    "MATEMATICA",
}

print("Repo root:", REPO_ROOT)
print("Raw path:", RAW_PATH)
print("Output path:", OUTPUT_PATH)

Repo root: /scratch/workspace/emma/standard-project-m2-mystery-machine
Raw path: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/raw/polizia_di_stato/polizia_di_stato_parsed.jsonl
Output path: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/polizia_di_stato_clean/polizia_di_stato_clean.jsonl


## Helpers

In [12]:
def read_jsonl(path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def write_jsonl(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def clean_text(x):
    if x is None or pd.isna(x):
        return ""
    text = str(x)
    text = text.replace("\u00a0", " ")
    text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()


def normalize_for_dedup(text):
    text = clean_text(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\sàèéìòóù]", "", text, flags=re.UNICODE)
    return text.strip()


def get_section(record):
    metadata = record.get("metadata", {})
    section = metadata.get("section", record.get("section", None))
    if section is None:
        return None
    return clean_text(section).upper()


def section_is_kept(section):
    if section is None:
        return True

    section = clean_text(section).upper()
    return section in KEEP_SECTIONS


OPTION_LINE_RE = re.compile(
    r"(?m)^\s*([A-E])[\).]\s*(.*?)(?=^\s*[A-E][\).]\s*|\Z)",
    re.DOTALL,
)

BAD_EXACT_OPTION_VALUES = {"", ".", "-", "—", "–", "A", "I", "T"}


def remove_isolated_watermark_lines(text):
    text = clean_text(text)
    lines = []
    for line in text.splitlines():
        stripped = line.strip()
        if stripped in {"A", "I", "T"}:
            continue
        lines.append(line)
    return clean_text("\n".join(lines))


def clean_option_text(text):
    text = remove_isolated_watermark_lines(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_question_text(text):
    text = remove_isolated_watermark_lines(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_prompt(prompt):
    prompt = clean_text(prompt)
    matches = list(OPTION_LINE_RE.finditer(prompt))

    if not matches:
        return clean_question_text(prompt), {}

    question = prompt[:matches[0].start()].strip()
    options = {}

    for match in matches:
        label = match.group(1).strip().upper()
        value = clean_option_text(match.group(2))
        options[label] = value

    return clean_question_text(question), options


def extract_question_options(record):
    if "prompt" in record:
        question, options = parse_prompt(record["prompt"])
    else:
        question, options = "", {}

    if not question:
        for key in ["question", "domanda", "stem"]:
            if key in record:
                question = clean_question_text(record[key])
                break

    option_field_candidates = {
        "A": ["A", "option_a", "answer_a", "a"],
        "B": ["B", "option_b", "answer_b", "b"],
        "C": ["C", "option_c", "answer_c", "c"],
        "D": ["D", "option_d", "answer_d", "d"],
        "E": ["E", "option_e", "answer_e", "e"],
    }

    for label, keys in option_field_candidates.items():
        if label in options and options[label]:
            continue
        for key in keys:
            if key in record:
                value = clean_option_text(record[key])
                if value:
                    options[label] = value
                break

    if "options" in record:
        raw_options = record["options"]
        if isinstance(raw_options, dict):
            for label, value in raw_options.items():
                label = str(label).strip().upper()
                if label in VALID_ANSWERS and label not in options:
                    options[label] = clean_option_text(value)

        elif isinstance(raw_options, list):
            for idx, value in enumerate(raw_options):
                label = chr(ord("A") + idx)
                if label in VALID_ANSWERS and label not in options:
                    options[label] = clean_option_text(value)

    return question, dict(sorted(options.items()))


def get_answer(record):
    answer = clean_text(
        record.get("answer", record.get("correct_answer", record.get("label", "")))
    ).upper()
    answer = answer.replace(")", "").replace(".", "").strip()
    return answer


def make_prompt(question, options):
    labels = sorted(options.keys())
    option_lines = [f"{label}) {options[label]}" for label in labels]
    return f"{question}\n\n" + "\n".join(option_lines)


def row_is_malformed(question, options, answer):
    if not question or len(question) < 8:
        return True, "bad_question"

    if answer not in VALID_ANSWERS:
        return True, "invalid_answer"

    # Keep 3-option questions, but remove obviously broken A/B-only examples.
    if len(options) < 3:
        return True, "too_few_options"

    if answer not in options:
        return True, "answer_option_missing"
    
    expected_labels = [chr(ord("A") + i) for i in range(len(options))]
    actual_labels = sorted(options.keys())

    if actual_labels != expected_labels:
        return True, "non_contiguous_option_labels"

    for label, value in options.items():
        if label not in VALID_ANSWERS:
            return True, "invalid_option_label"

        if clean_text(value) in BAD_EXACT_OPTION_VALUES:
            return True, "bad_option_text"

        if len(clean_text(value)) < 2:
            return True, "option_too_short"

    option_values = [normalize_for_dedup(v) for v in options.values()]
    if len(option_values) != len(set(option_values)):
        return True, "duplicate_options"

    full_text = question + " " + " ".join(options.values())
    if full_text.count("�") >= 2:
        return True, "replacement_characters"

    return False, None


def validate_record(record):
    assert set(record.keys()) >= {
        "id",
        "dataset",
        "language",
        "split",
        "task_type",
        "prompt",
        "answer",
        "metadata",
    }

    assert record["dataset"] == DATASET
    assert record["language"] == LANGUAGE
    assert record["split"] == SPLIT
    assert record["task_type"] == "mcq"
    assert record["answer"] in VALID_ANSWERS
    assert record["prompt"].strip()
    assert "\n\nA) " in record["prompt"]
    assert "\nB) " in record["prompt"]
    assert record["answer"] + ")" in record["prompt"]
    assert isinstance(record["metadata"]["source_idx"], int)

## Load

In [13]:
raw_records = read_jsonl(RAW_PATH)
df_raw = pd.DataFrame(raw_records)

print("Loaded rows:", len(df_raw))
print("Columns:", df_raw.columns.tolist())

sections = [get_section(record) for record in raw_records]
section_counts = pd.Series(sections).value_counts(dropna=False)

print("\nSection counts:")
display(section_counts.to_frame("n_examples"))

print("\nExample raw record:")
print(json.dumps(raw_records[0], ensure_ascii=False, indent=2)[:3000])

Loaded rows: 5652
Columns: ['id', 'dataset', 'language', 'task_type', 'prompt', 'answer', 'metadata']

Section counts:


,n_examples
INGLESE,2507
NaN,1906
MATEMATICA,1239



Example raw record:
{
  "id": "polizia_di_stato_italian_0",
  "dataset": "polizia_di_stato",
  "language": "italian",
  "task_type": "mcq",
  "prompt": "Il simbolo chimico del Magnesio?\n\nA) Se\nB) k\nC) Pb\nD) St\nE) Mg",
  "answer": "E",
  "metadata": {
    "source": "polizia_di_stato",
    "section": null,
    "original_id": 1
  }
}


In [14]:
def build_clean_dataframe(raw_records, section_filter_fn, source_name):
    stats = Counter()
    clean_rows = []

    for source_idx, record in enumerate(raw_records):
        section = get_section(record)

        if not section_filter_fn(section):
            stats[f"removed_section:{section}"] += 1
            continue

        question, options = extract_question_options(record)
        answer = get_answer(record)

        bad, reason = row_is_malformed(question, options, answer)
        if bad:
            stats[f"removed_{reason}"] += 1
            continue

        prompt = make_prompt(question, options)

        metadata = dict(record.get("metadata", {}))
        metadata.update({
            "source": "polizia_di_stato",
            "section": section,
            "original_id": record.get("id", record.get("original_id", source_idx)),
            "source_idx": source_idx,
            "n_options": len(options),
        })

        clean_rows.append({
            "source_idx": source_idx,
            "section": section,
            "question": question,
            "options": options,
            "prompt": prompt,
            "answer": answer,
            "metadata": metadata,
        })

    df_out = pd.DataFrame(clean_rows)
    return df_out, stats

## Deduplicate and filter

In [22]:
def deduplicate_dataframe(df_in):
    df_out = df_in.copy()
    df_out["dedup_key"] = df_out["prompt"].apply(normalize_for_dedup)

    labels_by_key = defaultdict(set)
    for _, row in df_out.iterrows():
        labels_by_key[row["dedup_key"]].add(row["answer"])

    conflicting_keys = {key for key, labels in labels_by_key.items() if len(labels) > 1}

    n_before = len(df_out)
    df_out = df_out[~df_out["dedup_key"].isin(conflicting_keys)].copy()
    n_after_conflicts = len(df_out)

    df_out = df_out.drop_duplicates(subset=["dedup_key", "answer"]).copy()
    n_after_exact = len(df_out)

    df_out = df_out.reset_index(drop=True)

    dedup_stats = {
        "before": n_before,
        "removed_conflicting_duplicates": n_before - n_after_conflicts,
        "removed_exact_duplicates": n_after_conflicts - n_after_exact,
        "after": len(df_out),
    }

    return df_out, dedup_stats

In [16]:
df, stats = build_clean_dataframe(
    raw_records=raw_records,
    section_filter_fn=section_is_kept,
    source_name="polizia_di_stato",
)

print("Raw rows:", len(raw_records))
print("After section filtering + malformed removal:", len(df))
print("\nRemoval stats:")
for key, value in sorted(stats.items()):
    print(f"{key}: {value}")

df, dedup_stats = deduplicate_dataframe(df)

print("\nDedup stats:")
for key, value in dedup_stats.items():
    print(f"{key}: {value}")

display(df.head())

Raw rows: 5652
After section filtering + malformed removal: 1756

Removal stats:
removed_duplicate_options: 12
removed_non_contiguous_option_labels: 125
removed_option_too_short: 12
removed_section:INGLESE: 2507
removed_section:MATEMATICA: 1239
removed_too_few_options: 1

Dedup stats:
before: 1756
removed_conflicting_duplicates: 0
removed_exact_duplicates: 0
after: 1756


,source_idx,section,question,options,prompt,answer,metadata,dedup_key
0,2,None,Il tricheco è possibile trovarlo:,"{'A': 'nella prateria', 'B': 'nella savana', '...",Il tricheco è possibile trovarlo:\n\nA) nella ...,D,"{'source': 'polizia_di_stato', 'section': None...",il tricheco è possibile trovarlo a nella prate...
1,3,None,Indicare tra questi animali quale non è un car...,"{'A': 'Corvo', 'B': 'Faina A', 'C': 'Ghepardo'...",Indicare tra questi animali quale non è un car...,E,"{'source': 'polizia_di_stato', 'section': None...",indicare tra questi animali quale non è un car...
2,4,None,Una stalagmite è:,{'A': 'Un deposito di zolfo che a forma di con...,Una stalagmite è:\n\nA) Un deposito di zolfo c...,D,"{'source': 'polizia_di_stato', 'section': None...",una stalagmite è a un deposito di zolfo che a ...
3,5,None,Il termine OGM indicNa:,{'A': 'il patrimonio genetico di un individuo'...,Il termine OGM indicNa:\n\nA) il patrimonio ge...,C,"{'source': 'polizia_di_stato', 'section': None...",il termine ogm indicna a il patrimonio genetic...
4,6,None,Indicare tra i seguenti cosa non fa parte dell...,"{'A': 'Rotula', 'B': 'Scapola', 'C': 'Lingua',...",Indicare tra i seguenti cosa non fa parte dell...,C,"{'source': 'polizia_di_stato', 'section': None...",indicare tra i seguenti cosa non fa parte dell...


In [17]:
print("Final rows:", len(df))

print("\nCounts by section:")
display(df["section"].value_counts(dropna=False).sort_index().to_frame("n_examples"))

print("\nAnswer distribution:")
display(df["answer"].value_counts().sort_index().to_frame("n_examples"))

print("\nNumber of options:")
display(df["options"].apply(len).value_counts().sort_index().to_frame("n_examples"))

print("\nExamples:")
for _, row in df.head(3).iterrows():
    print("=" * 80)
    print("source_idx:", row["source_idx"])
    print("section:", row["section"])
    print("answer:", row["answer"])
    print(row["prompt"])

Final rows: 1756

Counts by section:


,n_examples
section,
None,1756



Answer distribution:


,n_examples
answer,
A,376
B,352
C,351
D,358
E,319



Number of options:


,n_examples
options,
4,44
5,1712



Examples:
source_idx: 2
section: None
answer: D
Il tricheco è possibile trovarlo:

A) nella prateria
B) nella savana
C) nella foresta tropicale
D) in ambiente polare
E) nella tundra
source_idx: 3
section: None
answer: E
Indicare tra questi animali quale non è un carnivoro?

A) Corvo
B) Faina A
C) Ghepardo
D) Iena
E) Cervo
source_idx: 4
section: None
answer: D
Una stalagmite è:

A) Un deposito di zolfo che a forma di cono con punta verso il basso
B) Un deposito calcareo a forma di cono con punta verso il basso
C) Un deposito silicico a forma di cono con punta verso l'alto
D) Un deposito calcareo a forma di cono che si innalza dal pavimento
E) Una sorgente sotterranea termale


In [18]:
## Build final records

records = []

for idx, row in df.iterrows():
    record = {
        "id": f"polizia_di_stato_clean_italian_{idx}",
        "dataset": DATASET,
        "language": LANGUAGE,
        "split": SPLIT,
        "task_type": "mcq",
        "prompt": row["prompt"],
        "answer": row["answer"],
        "metadata": row["metadata"],
        "source_id": f"polizia_di_stato_{row['source_idx']}",
    }
    records.append(record)

print("Built records:", len(records))
print(json.dumps(records[0], ensure_ascii=False, indent=2))

Built records: 1756
{
  "id": "polizia_di_stato_clean_italian_0",
  "dataset": "polizia_di_stato_clean",
  "language": "italian",
  "split": "all",
  "task_type": "mcq",
  "prompt": "Il tricheco è possibile trovarlo:\n\nA) nella prateria\nB) nella savana\nC) nella foresta tropicale\nD) in ambiente polare\nE) nella tundra",
  "answer": "D",
  "metadata": {
    "source": "polizia_di_stato",
    "section": null,
    "original_id": "polizia_di_stato_italian_2",
    "source_idx": 2,
    "n_options": 5
  },
  "source_id": "polizia_di_stato_2"
}


In [19]:
## Save and validate

ids = [record["id"] for record in records]
assert len(ids) == len(set(ids)), "Duplicate IDs found."

prompt_keys = set()
for record in records:
    validate_record(record)
    key = normalize_for_dedup(record["prompt"])
    assert key not in prompt_keys, "Duplicate prompt found."
    prompt_keys.add(key)

write_jsonl(records, OUTPUT_PATH)
print(f"Saved {len(records)} records to:")
print(OUTPUT_PATH)

loaded = read_jsonl(OUTPUT_PATH)
assert len(loaded) == len(records)

for record in loaded:
    validate_record(record)

print("Reload validation passed.")

Saved 1756 records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/polizia_di_stato_clean/polizia_di_stato_clean.jsonl
Reload validation passed.


In [20]:
bad_records = []

for record in records:
    if "\nB) " not in record["prompt"]:
        bad_records.append(record)

print("Bad records without B):", len(bad_records))

for record in bad_records[:10]:
    print("=" * 80)
    print("id:", record["id"])
    print("answer:", record["answer"])
    print("metadata:", record["metadata"])
    print(record["prompt"])

Bad records without B): 0


In [21]:
print("Final summary")
print("-------------")
print(f"Raw rows:       {len(raw_records)}")
print(f"Saved records:  {len(records)}")
print(f"Output path:    {OUTPUT_PATH}")

print("\nSection counts:")
display(df["section"].value_counts(dropna=False).sort_index().to_frame("n_examples"))

print("\nAnswer distribution:")
display(pd.Series([record["answer"] for record in records]).value_counts().sort_index().to_frame("n_examples"))

print("\nOption-count distribution:")
display(df["options"].apply(len).value_counts().sort_index().to_frame("n_examples"))

Final summary
-------------
Raw rows:       5652
Saved records:  1756
Output path:    /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/polizia_di_stato_clean/polizia_di_stato_clean.jsonl

Section counts:


,n_examples
section,
None,1756



Answer distribution:


,n_examples
A,376
B,352
C,351
D,358
E,319



Option-count distribution:


,n_examples
options,
4,44
5,1712
